# 標準住宅の外皮性能

床断熱・基礎断熱の等級4～7について、面積と層構成からUA・ηAC・ηAHを計算します。窓のηdは枠・ガラス・付属部材から算定でき、仕様が未入力の窓では警告とともに季節別の不利側を採用します。

In [ ]:
%pip -q install git+https://github.com/iguchi-lab/vtsimnx.git@e283d30af2073343980805ea07e6741cf60c34d7

In [ ]:
import json
from pathlib import Path
import pandas as pd
import vtsimnx as vt

model = "foundation5"  # 床断熱は default, grade5～grade7。基礎断熱は foundation4～foundation7。
region = 6
house = vt.standard_house
types, layers = house.surface_types[model], house.layers[model]
input_data = {"surfaces": [
    {**row, "layers": layers.get(types[row["surface"]].get("layer_key"))}
    for row in house.surface_rows[model]
]}
# 室温計算用の input_data を組み立てた場合は、その辞書を直接使用できる。
path = Path(f"{model}_region{region}_rating.json")
if not path.exists():
    assessment = house.rating_template(model, region=region)
    path.write_text(json.dumps(assessment, ensure_ascii=False, indent=2), encoding="utf-8")
print(path.resolve())

JSONの窓に、例えば `frame_type` = 金属製など、`glass_type` = 一般複層、`attachment` = なし と入力するとηdは0.632です。既存JSONにもこの3項目を追加できます。仕様の空欄は不利側を採用し、部位別結果に採用値を表示します。直接入力したηdも使えます。開口部U値と土間外周の長さ・ψには設計値が必要です。[選択肢と計算方法](https://github.com/iguchi-lab/vtsimnx/blob/e283d30af2073343980805ea07e6741cf60c34d7/docs/envelope_rating.md)

In [ ]:
assessment = json.loads(path.read_text(encoding="utf-8"))
missing = []
for part in assessment["parts"]:
    name = part["keys"][0]
    if part["kind"] in {"window", "door"} and part.get("U") is None:
        missing.append((name, "U", "開口部全体の熱貫流率 [W/(m²·K)]"))
for line in assessment["lines"]:
    for field, label in [("length", "長さ [m]"), ("psi", "線熱貫流率 [W/(m·K)]"), ("H", "温度差係数")]:
        if line.get(field) is None:
            missing.append((line["name"], field, label))

if missing:
    print(f"{path.name} に以下の設計値を入力して保存し、このセルを再実行してください。")
    display(pd.DataFrame(missing, columns=["部位", "JSONの項目", "必要な設計値"]))
else:
    result = vt.envelope_performance(input_data, assessment)
    display(pd.DataFrame([{k: v for k, v in result.items() if k != "parts"}]))
    display(pd.DataFrame(result["parts"]))